# Interfaz de Python/NumPy con código en C/C++ y Fortran

## Motivación

Ya viste, desde el teaser de difusión hasta el notebook 03, que Python puro es lento y que NumPy soluciona esto porque, por debajo, delega el cálculo a código C/Fortran ya compilado. Esta notebook te muestra cómo hacer *tú mismo* ese mismo truco: llamar código compilado en C, C++ o Fortran directamente desde Python.

¿Por qué querrías hacer esto?
* Ya existe código en C/C++/Fortran (a veces de hace décadas) que hace exactamente lo que necesitas, y no tiene sentido reescribirlo desde cero en Python.
* Escribiste (o vas a escribir) un cálculo numérico muy específico ("hotspot") que, después de perfilarlo (notebook 07), resulta ser el cuello de botella real de tu programa, y necesitas la velocidad de C/Fortran solo para esa parte.

La idea general: combinar la comodidad de escribir en Python de alto nivel, con el rendimiento de código compilado donde de verdad importa.

## Los pasos generales

Sin importar qué herramienta uses (las vas a ver todas abajo), el proceso es siempre el mismo:

1. Escribir una **interfaz** (una capa intermedia) entre Python/NumPy y el código en C/C++/Fortran — un "traductor" que sepa convertir datos de Python a datos que C/Fortran entienda, y viceversa.
2. **Compilar y enlazar** ese código para que el resultado sea un módulo que Python pueda importar con un `import` normal — esto normalmente se integra con el empaquetado que viste en el notebook 05 (`pyproject.toml`, `pip install`).

## ¿Qué herramienta uso, según el lenguaje?

No existe una única herramienta "oficial" — distintas herramientas se especializan en distintos lenguajes de origen:

| Lenguaje del código existente | Herramienta recomendada |
|---|---|
| Fortran | **f2py** |
| C / C++ / CUDA | **Cython** |
| C++ moderno (clases, plantillas) | **pybind11** o **nanobind** |

Otras opciones que vas a encontrar en código legado, pero que no cubrimos en detalle aquí: Boost.Python (para C++), SWIG (un generador de interfaces genérico), o usar directamente la API de C de Python (el camino más difícil y menos recomendado, salvo casos muy específicos).

**Para orientarte:** si tienes código Fortran ya escrito, usa f2py. Si tienes código C o quieres escribir un cálculo optimizado desde cero pero con sintaxis parecida a Python, usa Cython (ya la usaste en los notebooks 03 y 04, sin necesariamente saberlo del todo). Si tienes una librería de C++ moderna con clases, usa pybind11 o nanobind.

## `f2py`: interfaz con código Fortran

`f2py` viene incluido con NumPy — no hace falta instalar nada extra.

* Escanea el código Fortran y genera automáticamente los archivos de "firma" (`.pyf`) necesarios para la interfaz.
* Se encarga automáticamente de convertir tipos de datos y manejar arreglos no contiguos — tú casi no tienes que escribir código de interfaz a mano, a diferencia de Cython o pybind11.
* Dos formas de usarlo: llamando directamente al ejecutable `f2py`, o integrándolo en un sistema de build más completo (`meson`, `cmake`) para proyectos más grandes.

### Ejemplo: compilar código Fortran a un módulo de Python

Supongamos que tenemos este archivo Fortran, `fib.f90`, que calcula los primeros `n` números de la secuencia de Fibonacci:

```fortran
subroutine fib(a,n)
  integer, intent(in) :: n
  integer(kind=8), intent(out) :: a(n)
  do i=1,n
     if (i.eq.1) then
        a(i) = 0
     elseif (i.eq.2) then
        a(i) = 1
     else
        a(i) = a(i-1) + a(i-2)
     endif
  enddo
end
```

Con un solo comando en la terminal, generamos un módulo de Python importable:

```bash
f2py -c fib.f90 -m fib
```

Esto genera un archivo compilado (por ejemplo `fib.cpython-311-....so`) que ya puedes importar directamente:

```python
from fib import fib
a = fib.fib(16)
print(a[-1])   # 610
```

**Nota:** este ejemplo necesita un compilador de Fortran instalado (`gfortran`, el mismo de la Tarea 1 del notebook 01). Si en tu laptop no tienes `gfortran` disponible, puedes seguir el ejemplo como lectura — lo importante es entender el flujo: código Fortran → un comando de `f2py` → módulo de Python importable, sin escribir ninguna línea de código de interfaz a mano. Puedes encontrar este ejemplo completo en la carpeta [`examples/f2py`](../examples/f2py) de este repositorio.

### Empaquetar código Fortran con `meson`

Para un proyecto más grande (más de un archivo Fortran), conviene integrar la compilación dentro de un `pyproject.toml`, usando `meson` como sistema de build:

```toml
[build-system]
requires = ['meson-python', 'numpy']
build-backend = 'mesonpy'

[project]
name = 'f2py_example'
version = '1.0.0'
```

`meson` lee la configuración detallada de compilación desde un archivo separado, `meson.build` (ver el ejemplo completo en `examples/f2py`). Una vez configurado, instalar y usar el paquete es exactamente igual a lo que viste en el notebook 05:

```bash
pip install .
python -c "from f2py_example import fib; a=fib.fib(16); print(a[-1])"
# 610
```

## Cython: interfaz con código C/C++

Ya usaste Cython en los notebooks 03 y 04 para *acelerar* código Python (compilarlo directamente). Ahora lo vas a ver desde otro ángulo: usar Cython como **puente** hacia código C que ya existe, escrito por separado.

**La estrategia general:** escribes una extensión de Cython (un archivo `.pyx`) que define funciones de Python cuyo cuerpo simplemente llama al código C externo.

### Ejemplo paso a paso

Un header de C con la declaración de una función:
```c
/* c_hello.h */
void hello(void);
```

Su implementación en C:
```c
/* c_hello.c */
#include <stdio.h>
#include "c_hello.h"

void hello(void) {
    printf("Hello World!\n");
}
```

El "puente" en Cython, que le dice a Python que esta función de C existe y la expone como una función normal de Python:
```cython
# hello.pyx
cdef extern from "c_hello.h":
    void hello()

def say_hello():
    hello()
```

Y el `pyproject.toml` que junta todo, indicándole a `setuptools` que compile tanto el `.pyx` como el `.c`:
```toml
[build-system]
requires = ["setuptools", "cython"]
build-backend = "setuptools.build_meta"

[project]
name = "foobar"
version = "0.0.1"

[tool.setuptools]
ext-modules = [
  {name = "foobar.hello", sources = ["src/foobar/hello.pyx", "src/foobar/c_hello.c"]}
]
```

Una vez instalado con `pip install .`, en Python simplemente harías `from foobar.hello import say_hello; say_hello()`, y por debajo se ejecuta la función escrita en C. Ver el ejemplo completo en [`examples/cython/c_interface`](../examples/cython).

## Pasar arreglos de NumPy a código en C

El caso más común en HPC no es solo "llamar una función de C", sino pasarle un arreglo de NumPy completo para que lo procese, y recibir el resultado de vuelta como otro arreglo de NumPy — dejando que Python/NumPy se encarguen de reservar y liberar la memoria, no el código C. Ver el ejemplo completo en [`examples/cython/c_numpy`](../examples/cython).

### Consejos prácticos

**En la capa de Cython:**
* Revisa cuidadosamente los tipos de datos que se pasan (`int` vs `long`, `float32` vs `float64`, etc.) — un tipo incorrecto es una fuente común de bugs difíciles de rastrear.
* Se pasa el puntero a los datos del arreglo de NumPy (el atributo `.data`) hacia la función de C.
* Cualquier memoria temporal conviene reservarla como un arreglo de NumPy en la capa de Cython, no dentro del código C.
* Antes de pasar un arreglo a C, asegúrate de que sea **contiguo** en memoria (recuerda el notebook 03) — si es una vista no contigua, hay que copiarla primero.

**En la capa de C:**
* Escribe la extensión sin estado interno (*stateless*) y cuidado con fugas de memoria (*memory leaks*).
* Aquí puedes aplicar cualquier técnica de optimización que conozcas: hilos con OpenMP (que, a diferencia de Python, no están limitados por el GIL — lo verás en el notebook de paralelismo), vectorización, optimización de caché, etc.
* Los flags de optimización del compilador se configuran según el sistema de build que uses (por ejemplo, dentro del `pyproject.toml` con `setuptools`).

## Usar una librería de C ya compilada (`.so`) desde Cython

A veces no tienes el código fuente en C, sino una librería ya compilada (un archivo `.so` en Linux/Mac, o `.dll` en Windows) — por ejemplo:
* Una librería de terceros para la que no existen *bindings* de Python.
* Código legado que quieres usar desde Python sin reescribirlo.
* Código CUDA, compilado primero a un `.so` con `nvcc` de forma independiente.

Ver el ejemplo completo en [`examples/cython/c_interface_shared_object`](../examples/cython).

**Consejos prácticos:**
* Tu script de instalación necesita saber dónde está la librería y sus archivos de cabecera (`.h`). Las formas más comunes en HPC: una variable de entorno (como `MKL_ROOT`, `GSL_HOME`, `FFTW_HOME`, típico en módulos de entorno de un clúster), un argumento al instalar, o una ubicación estándar del sistema.
* Al distribuir, conviene incluir la ruta a la librería directamente en el ejecutable (usando `RPATH`), para no depender de que el usuario configure `LD_LIBRARY_PATH` a mano.
* Para código CUDA, suele ser más simple compilarlo primero a una librería `.so` por separado, en vez de integrar `nvcc` directamente en el script de instalación.

## Cython y C++

Cython también soporta muchas características de C++ orientado a objetos: clases, plantillas (*templates*), sobrecarga de funciones. El caso de uso típico es escribir una clase de Python que "envuelve" (*wrap*) una clase de C++ existente.

No vamos a cubrir esto en profundidad — si lo necesitas en el futuro, la documentación oficial es un buen punto de partida: http://cython.readthedocs.io/en/latest/src/userguide/wrapping_CPlusPlus.html

### Actividad: Cython como puente hacia `libc` y la STL de C++

Una de las cosas más prácticas de Cython es que ya trae interfaces listas hacia la librería estándar de C (`libc`) y hacia la STL de C++ (contenedores como `vector`), sin que tengas que escribir tú el puente. Vamos a probarlo directamente.

In [ ]:
%load_ext Cython

In [ ]:
%%cython
from libc.math cimport sin

print(sin(1.0))

Esa línea llamó directamente a la función `sin` de la librería matemática de C (no a `math.sin` ni a `np.sin` de Python) — sin escribir ningún archivo `.c` ni `.h` a mano.

**Actividad:** modifica la celda de arriba para usar `cos` en vez de `sin` (vas a necesitar importar `from libc.math cimport cos`), y compara el resultado con `import math; math.cos(1.0)` — deberían coincidir.

In [ ]:
%%cython
# distutils: language = c++
from libcpp.vector cimport vector

cdef vector[int] v = range(10)
print(v)

Esta celda creó un `vector` de C++ (el contenedor más usado de la STL) directamente desde Cython, inicializándolo con un `range` de Python. La línea `# distutils: language = c++` es necesaria para decirle a Cython que compile como C++ y no como C — sin ella, esta celda fallaría.

## pybind11

**pybind11** es una librería *header-only* (no hay que compilarla ni instalarla como librería separada, solo incluirla) para crear interfaces con código C++ moderno.

Puntos destacados: soporte para la STL de C++, iteradores, clases e herencia, punteros inteligentes (*smart pointers*), semántica de movimiento, conversión directa entre arreglos de NumPy y estructuras de C++, e integración con la librería de álgebra lineal Eigen.

Es compatible con varios sistemas de build: `cmake` (el más común), `meson`, `cppimport`, `setuptools`. Documentación: https://pybind11.readthedocs.io/en/latest/

### Ejemplo de integración con `setuptools`

```toml
[build-system]
requires = ["setuptools", "pybind11"]
build-backend = "setuptools.build_meta"
```

```python
# setup.py
from setuptools import setup
from pybind11.setup_helpers import Pybind11Extension, build_ext

ext_modules = [
    Pybind11Extension(
        "pybex",
        ["src/pybex.cpp"],
    ),
]

setup(
    cmdclass={"build_ext": build_ext},
    ext_modules=ext_modules
)
```

Ver el ejemplo completo con NumPy en [`examples/pybind11`](../examples/pybind11) de este repositorio.

## nanobind

**nanobind** es, en palabras simples, una versión más liviana de pybind11, hecha por el mismo autor, con mejoras significativas de rendimiento tanto al compilar como al ejecutar. Se puede construir y empaquetar cómodamente con `cmake` y `scikit-build-core`.

Ver el ejemplo completo en [`examples/nanobind`](../examples/nanobind) de este repositorio. Documentación: https://nanobind.readthedocs.io/en/latest/

## Resumen

* Interfazar Python con C/C++/Fortran te permite combinar la comodidad de Python con el rendimiento de código compilado, sin reescribir todo tu proyecto.
* La elección de herramienta depende del lenguaje de origen: **f2py** para Fortran, **Cython** para C/C++/CUDA, **pybind11**/**nanobind** para C++ moderno.
* En todos los casos, el flujo es el mismo: escribir una interfaz → compilar → obtener un módulo importable desde Python, normalmente integrado con el empaquetado del notebook 05.
* No necesitas dominar las cuatro herramientas — con que sepas que existen y cuál corresponde a cada situación, ya puedes investigar la que te haga falta el día que la necesites.

## Autoevaluación

Responde con tus propias palabras (edita esta celda):

1. Tienes un código en Fortran de los años 90 que hace un cálculo numérico muy específico. ¿Qué herramienta de esta notebook usarías para llamarlo desde Python, y por qué?
2. ¿Cuál es la diferencia entre usar Cython para *acelerar* código Python (como en los notebooks 03 y 04) y usar Cython como *puente* hacia código C ya existente (como en esta notebook)?
3. En la actividad de `libc.math`, ¿por qué la celda del `vector` de C++ necesitó la línea `# distutils: language = c++` y la de `sin` no?
4. Si tuvieras que envolver una librería de C++ moderna, con varias clases, ¿preferirías Cython o pybind11? ¿Por qué?

**Tus respuestas:**

_(escribe aquí)_